<a href="https://colab.research.google.com/github/ksuplee/tensorflow-nlp-tutorial/blob/main/13_AI_Agent/13_03_LLM_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13_03 LLM 에이전트 : ReAct 루프와 함수 호출(Function Calling)

**학습 목표**
- 에이전트의 **사고(Reasoning) → 행동(Action) → 관찰(Observation)** 루프를 직접 구현해 본다.
- 에이전트가 **도구(tool)** 를 호출해 여러 단계를 자율적으로 수행하는 과정을 관찰한다.
- LLM API의 **함수 호출(Function Calling)** 개념을 이해한다.

> ※ 본 실습은 **개념·데모 수준**입니다. 외부 API 키 없이도 1~2단계가 실행되며, 심화 구현은 **'AI 에이전트 개발'** 교과목에서 다룹니다.

## 1. 도구(Tools) 정의

에이전트가 호출할 수 있는 함수들을 정의합니다. 각 함수는 하나의 **도구**입니다. 앞서 배운 **RAG(검색)도 하나의 도구**로 볼 수 있습니다(`search`).

In [ ]:
import re

def calculator(expression: str) -> str:
    """사칙연산 문자열을 계산한다. 예: '25*4' -> '100'"""
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "오류: 허용되지 않은 문자"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"오류: {e}"

_WEATHER_DB = {"서울": "맑음, 28도", "부산": "흐림, 26도", "제주": "비, 24도"}
def get_weather(city: str) -> str:
    """도시의 현재 날씨(모의 데이터)를 반환한다."""
    return _WEATHER_DB.get(city, "정보 없음")

_KB = {
    "트랜스포머": "트랜스포머는 2017년 발표된 셀프 어텐션 기반 모델이다.",
    "bert": "BERT는 양방향 인코더로 사전학습된 언어모델이다.",
}
def search(query: str) -> str:
    """간단한 지식베이스에서 관련 문장을 찾는다(RAG의 검색도 도구의 하나)."""
    for k, v in _KB.items():
        if k in query.lower():
            return v
    return "검색 결과 없음"

# 도구 레지스트리: 이름 -> 함수
TOOLS = {"calculator": calculator, "get_weather": get_weather, "search": search}
print("등록된 도구:", list(TOOLS.keys()))

## 2. ReAct 루프 직접 구현 (API 없이)

실제 에이전트에서는 **LLM이 다음 행동을 '추론'**합니다. 여기서는 원리를 눈으로 확인하기 위해 LLM 자리에 **간단한 규칙 기반 플래너**(`plan_next_action`)를 넣어 루프의 동작을 모사합니다.

In [ ]:
CITIES = ["서울", "부산", "제주", "인천", "대구"]
OP = {"곱하기": "*", "더하기": "+", "빼기": "-", "나누기": "/"}

def plan_next_action(goal, history):
    """LLM을 대신하는 규칙 기반 플래너. 다음에 호출할 (도구, 인자, 사고)를 반환한다."""
    used = {h["tool"] for h in history}
    # 1) 날씨 요청
    if "날씨" in goal and "get_weather" not in used:
        city = next((c for c in CITIES if c in goal), "서울")
        return ("get_weather", city, f"'{city}'의 날씨를 확인해야 한다.")
    # 2) 사칙연산 요청
    m = re.search(r"(\d+)\s*(곱하기|더하기|빼기|나누기)\s*(\d+)", goal)
    if m and "calculator" not in used:
        expr = f"{m.group(1)}{OP[m.group(2)]}{m.group(3)}"
        return ("calculator", expr, f"'{expr}' 를 계산해야 한다.")
    # 3) 더 할 일이 없으면 종료
    return (None, None, "필요한 정보를 모두 모았으니 최종 답변을 정리한다.")

def react_agent(goal, max_steps=5):
    """사고 -> 행동 -> 관찰 루프를 반복하며 목표를 수행한다."""
    history = []
    print(f"[목표] {goal}\n")
    for step in range(1, max_steps + 1):
        tool, arg, thought = plan_next_action(goal, history)
        print(f"[{step}] 사고(Thought): {thought}")
        if tool is None:
            break
        obs = TOOLS[tool](arg)
        history.append({"tool": tool, "arg": arg, "obs": obs})
        print(f"    행동(Action): {tool}({arg!r})")
        print(f"    관찰(Observation): {obs}\n")
    answer = " / ".join(f"{h['arg']} → {h['obs']}" for h in history)
    print(f"[최종 답변] {answer}")
    return answer

**실행:** 검색·계산 두 단계가 필요한 질문을 주면, 에이전트가 스스로 순서대로 도구를 호출합니다.

In [ ]:
_ = react_agent("25 곱하기 4는 얼마이고, 서울 날씨는 어때?")

> 💡 **관찰 포인트**: 한 번의 답이 아니라 **여러 단계(사고→행동→관찰)를 반복**하며 필요한 도구를 골라 호출하는 것이 에이전트의 핵심입니다.

## 3. OpenAI Function Calling (선택 · API 키 필요)

실제 LLM API에서는 모델이 **'어떤 함수를 어떤 인자로 호출할지'를 구조화된 JSON**으로 반환합니다. 모델은 **함수를 직접 실행하지 않고**, 호출을 **결정**만 합니다. 실제 실행은 개발자 코드가 담당합니다.

> 아래 셀은 `OPENAI_API_KEY`가 설정되어 있으면 실제 호출을, 없으면 개념 설명을 출력합니다.

In [ ]:
# (선택) 실제 호출을 원하면 먼저 설치: !pip install -q openai
import os, json

tools_schema = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "도시명"}},
            "required": ["city"],
        },
    },
}]

api_key = os.environ.get("OPENAI_API_KEY")
if api_key:
    from openai import OpenAI
    client = OpenAI()
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "서울 날씨 알려줘"}],
        tools=tools_schema,
    )
    tool_calls = resp.choices[0].message.tool_calls
    print("모델이 호출을 결정한 함수:")
    for c in tool_calls or []:
        print(" -", c.function.name, c.function.arguments)
else:
    print("OPENAI_API_KEY가 없어 개념만 설명합니다.\n")
    print("모델은 다음과 같이 '호출할 함수와 인자'를 JSON으로 반환합니다:")
    print(json.dumps({"name": "get_weather", "arguments": {"city": "서울"}}, ensure_ascii=False))
    print("→ 실제 실행은 개발자 코드가 담당하고, 그 결과를 다시 모델에 전달합니다.")

## 4. 정리

| 개념 | 요약 |
|------|------|
| ReAct | 사고→행동→관찰을 반복하는 에이전트 루프 |
| Tool(도구) | 검색(RAG)·계산·코드 실행·외부 API 등 |
| Function Calling | 모델이 호출할 함수·인자를 JSON으로 **결정**, 실행은 시스템이 담당 |

> 🔗 **연계**: 멀티 에이전트·플래닝·메모리·MCP·프로덕션 배포 등 심화는 **'AI 에이전트 개발'** 교과목에서 다룹니다.